# 🦏 Detecção de animais selvagens com YOLO26

Treina um detector de objetos capaz de identificar **quatro animais africanos**
em imagens e vídeo:

🐃 búfalo · 🐘 elefante · 🦏 rinoceronte · 🦓 zebra

**Modelo:** Ultralytics YOLO26
**Dataset:** African Wildlife (oficial da Ultralytics, baixa sozinho)
**Ambiente:** Google Colab com GPU

---

## ⚙️ Antes de começar

**Ative a GPU:** `Ambiente de execução → Alterar o tipo de ambiente de execução
→ Acelerador de hardware: GPU (T4) → Salvar`

Depois é só rodar as células em ordem. Nada precisa ser baixado ou configurado
à mão — o dataset vem junto com o pacote.

**Tempo estimado:** cerca de 25 a 35 minutos na GPU T4 gratuita.

---

## Roteiro

| Etapa | O que acontece |
|---|---|
| 1 | Instala o YOLO26 e confirma a GPU |
| 2 | Conhece o dataset African Wildlife |
| 3 | Treina o modelo (fine-tuning) |
| 4 | Avalia com métricas e matriz de confusão |
| 5 | Testa em imagens |
| 6 | Testa em vídeo |

---
# 1. Instalação e verificação

O pacote `ultralytics` traz o YOLO26 e todo o pipeline de treino, validação e
inferência. Uma linha só.

`ultralytics.checks()` imprime a versão do pacote, do PyTorch, o modelo da GPU
e a memória disponível — é a primeira coisa a conferir quando algo dá errado.

In [ ]:
!pip install -q ultralytics

import ultralytics
ultralytics.checks()

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"Memória : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print("\nTudo certo, pode seguir.")
else:
    print("[!] SEM GPU DETECTADA — o treino levaria horas.")
    print("    Ambiente de execução > Alterar o tipo de ambiente de execução")
    print("    > Acelerador de hardware: GPU (T4) > Salvar")
    print("    Depois rode esta célula novamente.")

---
# 2. O dataset African Wildlife

É um dos datasets oficiais da Ultralytics, então **não precisa baixar nada
manualmente**: basta escrever `african-wildlife.yaml` e o pacote busca os
arquivos sozinho (cerca de 100 MB) na primeira vez que o treino roda.

| | |
|---|---|
| **Imagens** | 1.504 no total |
| **Classes** | 4 — buffalo, elephant, rhino, zebra |
| **Treino** | 1.052 imagens |
| **Validação** | 225 imagens |
| **Teste** | 227 imagens |

As imagens vêm de reservas naturais da África do Sul. O dataset foi publicado
originalmente no Kaggle por Bianca Ferreira e adaptado pela Ultralytics.

## Como o dataset é descrito

Todo dataset de detecção no formato Ultralytics é definido por um arquivo YAML
que aponta as pastas e nomeia as classes. Vamos ler o arquivo que já vem
instalado com o pacote.

In [ ]:
import yaml
from pathlib import Path
from ultralytics.utils import ROOT

caminho = ROOT / "cfg" / "datasets" / "african-wildlife.yaml"
cfg = yaml.safe_load(open(caminho, encoding="utf-8"))

print(f"Arquivo: {caminho}\n")
print("Pastas:")
for chave in ["path", "train", "val", "test"]:
    if chave in cfg:
        print(f"  {chave:<7} {cfg[chave]}")

print("\nClasses:")
for i, nome in cfg["names"].items():
    print(f"  {i}  {nome}")

### Como os rótulos são guardados

Cada imagem tem um arquivo `.txt` de mesmo nome, com uma linha por animal:

```
<id_classe> <x_centro> <y_centro> <largura> <altura>
```

As quatro coordenadas são **normalizadas de 0 a 1**, ou seja, expressas como
fração do tamanho da imagem. Isso deixa o rótulo independente da resolução:
a mesma anotação vale se você redimensionar a foto.

Exemplo: `2 0.51 0.44 0.30 0.62` significa um **rinoceronte** (classe 2)
centrado quase no meio da imagem, ocupando 30% da largura e 62% da altura.

---
# 3. Treino

## Por que fine-tuning, e não treinar do zero

Os checkpoints do YOLO26 já passaram por **150 épocas de pré-treino no
Objects365** e depois foram ajustados no COCO. Os dois datasets contêm
`elephant` e `zebra` entre suas classes.

Ou seja: **o modelo já sabe reconhecer animais grandes** e onde eles começam e
terminam numa imagem. Nosso treino só precisa especializá-lo nessas quatro
espécies. Por isso 1.052 imagens bastam — treinar do zero exigiria dezenas de
milhares.

## Qual tamanho de modelo escolher

| Modelo | Parâmetros | mAP no COCO | Quando usar |
|---|---|---|---|
| `yolo26n.pt` | 2,4 M | 40,9 | Mais rápido, bom para testar o fluxo |
| `yolo26s.pt` | 9,5 M | 48,6 | **Escolha deste notebook** — bom equilíbrio |
| `yolo26m.pt` | 20,4 M | 53,1 | Mais preciso, treino mais demorado |

## Os parâmetros do treino

| Parâmetro | Valor | O que faz |
|---|---|---|
| `epochs=50` | 50 passadas | Quantas vezes o modelo vê o dataset inteiro |
| `imgsz=640` | 640×640 | Resolução de entrada; os animais são grandes, 640 basta |
| `batch=16` | 16 imagens | Quantas processa por vez. Reduza para 8 se faltar memória |
| `patience=15` | parada antecipada | Interrompe se o mAP não melhorar por 15 épocas |
| `seed=42` | semente fixa | Deixa o resultado reproduzível |
| `plots=True` | gráficos | Gera curvas, matriz de confusão e amostras automaticamente |

> O download do dataset acontece agora, na primeira execução. Se aparecer uma
> barra de progresso antes do treino começar, é isso.

In [ ]:
from ultralytics import YOLO

modelo = YOLO("yolo26s.pt")

resultados = modelo.train(
    data="african-wildlife.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    seed=42,
    plots=True,
    name="wildlife",
    exist_ok=True,
)

PESOS = "runs/detect/wildlife/weights/best.pt"
print("\nTreino concluído. Melhores pesos em:", PESOS)

### As amostras de treino

O Ultralytics salva um mosaico das imagens como o modelo as enxerga durante o
treino, já com as caixas desenhadas. Serve para conferir se os rótulos estão
alinhados com os animais — rótulo torto é a causa mais comum de treino ruim.

In [ ]:
from IPython.display import Image, display

display(Image("runs/detect/wildlife/train_batch0.jpg", width=900))

### As curvas de treino

O que olhar no gráfico abaixo:

- **box_loss / cls_loss / dfl_loss** devem cair de forma suave. `box_loss` mede
  o erro de posição da caixa; `cls_loss` mede o erro de qual animal é.
- **As curvas de validação** (linha de baixo) devem acompanhar as de treino. Se
  a de treino cair enquanto a de validação sobe, é overfitting: o modelo
  decorou as imagens em vez de aprender o padrão.
- **mAP50 e mAP50-95** devem subir e depois estabilizar. O platô é normal e
  indica que o modelo convergiu.

In [ ]:
display(Image("runs/detect/wildlife/results.png", width=1000))

---
# 4. Avaliação

Agora medimos o desempenho no conjunto de validação — imagens que o modelo
**nunca viu durante o treino**.

| Métrica | Pergunta que responde | O que significa estar baixa |
|---|---|---|
| **Precisão (P)** | Das caixas que apontei, quantas estavam certas? | O modelo inventa detecções |
| **Recall (R)** | Dos animais que existiam, quantos eu achei? | O modelo deixa animais passarem |
| **mAP50** | Acerto considerando 50% de sobreposição com a caixa real | Métrica mais generosa, a mais citada |
| **mAP50-95** | Média do mAP em 10 níveis de exigência (50% a 95%) | Mais rigorosa: mede também a **qualidade** da caixa |

Vale olhar as métricas **por classe**, não só a média. Uma classe muito abaixo
das outras indica um problema específico dela — não do modelo inteiro.

In [ ]:
from ultralytics import YOLO

modelo = YOLO(PESOS)
metricas = modelo.val(data="african-wildlife.yaml", split="val",
                      plots=True, name="wildlife_val", exist_ok=True)

caixa = metricas.box

print("\n" + "=" * 58)
print("RESULTADO GERAL")
print("=" * 58)
print(f"  Precisão ... {caixa.mp:.4f}")
print(f"  Recall ..... {caixa.mr:.4f}")
print(f"  mAP50 ...... {caixa.map50:.4f}")
print(f"  mAP50-95 ... {caixa.map:.4f}")

print("\n" + "=" * 58)
print("RESULTADO POR CLASSE")
print("=" * 58)
print(f"  {'animal':<14}{'P':>9}{'R':>9}{'mAP50':>9}{'mAP50-95':>11}")

linhas = []
for i, idx in enumerate(caixa.ap_class_index):
    nome = modelo.names[int(idx)]
    p, r, ap50, ap = caixa.class_result(i)
    linhas.append((nome, ap50))
    print(f"  {nome:<14}{p:>9.4f}{r:>9.4f}{ap50:>9.4f}{ap:>11.4f}")

print("\n" + "=" * 58)
if caixa.map50 >= 0.90:
    print("Excelente: mAP50 acima de 0,90. As quatro espécies são bem distintas")
    print("entre si, então esse patamar é esperado num treino saudável.")
elif caixa.map50 >= 0.75:
    print("Bom: mAP50 entre 0,75 e 0,90. Resultado utilizável, com erros")
    print("pontuais em imagens difíceis (animal distante ou parcialmente oculto).")
else:
    print("Abaixo do esperado. Tente aumentar 'epochs' ou usar yolo26m.pt.")

if linhas:
    pior = min(linhas, key=lambda x: x[1])
    melhor = max(linhas, key=lambda x: x[1])
    print(f"\nMelhor classe: {melhor[0]} ({melhor[1]:.3f})")
    print(f"Pior classe:   {pior[0]} ({pior[1]:.3f})")
print("=" * 58)

### Matriz de confusão

É o gráfico mais informativo da avaliação. Cada coluna é o que o animal
**realmente era**, cada linha é o que o modelo **achou que era**.

- **Diagonal forte** → o modelo separa bem as espécies
- **Valor fora da diagonal** → confusão entre duas espécies específicas
  (búfalo e rinoceronte são os candidatos mais prováveis: ambos escuros,
  corpulentos e de perfil parecido a distância)
- **Coluna `background`** → o modelo inventou um animal onde não havia nada
- **Linha `background`** → o modelo deixou um animal passar despercebido

A versão normalizada mostra proporções em vez de contagens, o que facilita a
comparação entre classes com números diferentes de exemplos.

In [ ]:
display(Image("runs/detect/wildlife_val/confusion_matrix_normalized.png", width=760))

### Curva Precisão × Recall

Cada linha é uma classe. Quanto mais a curva se aproxima do canto superior
direito, melhor: significa acertar muito (recall alto) sem inventar detecções
(precisão alta). A área sob a curva **é** o mAP50.

In [ ]:
display(Image("runs/detect/wildlife_val/PR_curve.png", width=760))

---
# 5. Teste em imagens

Rodando o modelo em imagens do conjunto de teste — que não participaram nem do
treino nem da validação.

O parâmetro `conf=0.5` descarta detecções com menos de 50% de confiança. Baixar
esse valor faz o modelo detectar mais animais, mas também aumenta os falsos
positivos. É o botão que equilibra recall contra precisão.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
from ultralytics import YOLO
from ultralytics.utils import SETTINGS

modelo = YOLO(PESOS)

# O Ultralytics baixa os datasets nessa pasta
raiz = Path(SETTINGS["datasets_dir"]) / "african-wildlife"
pasta_teste = raiz / "test" / "images"
if not pasta_teste.is_dir():
    pasta_teste = raiz / "valid" / "images"

arquivos = sorted(p for p in pasta_teste.iterdir()
                  if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
print(f"{len(arquivos)} imagens de teste em {pasta_teste}")

random.seed(42)
amostra = random.sample(arquivos, k=6)
saidas = modelo.predict(amostra, conf=0.5, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, res in zip(axes.ravel(), saidas):
    ax.imshow(res.plot()[:, :, ::-1])      # OpenCV usa BGR, matplotlib usa RGB
    ax.axis("off")
    achados = [f"{modelo.names[int(c)]} {f:.2f}"
               for c, f in zip(res.boxes.cls, res.boxes.conf)]
    ax.set_title(" | ".join(achados) if achados else "nada detectado", fontsize=11)

plt.tight_layout()
plt.show()

---
# 6. Teste em vídeo

Detectar em vídeo é o mesmo que detectar em imagem, repetido quadro a quadro.
A diferença é que usamos `model.track()` em vez de `model.predict()`.

**Por que rastrear.** O `track()` acrescenta um **ID persistente** a cada
animal. Sem ele, o modelo detecta "uma zebra" em cada quadro sem saber que é a
mesma zebra de antes. Com ID, dá para contar animais únicos, seguir trajetórias
e evitar contar o mesmo bicho várias vezes.

A célula abaixo monta um vídeo a partir das imagens de teste, para que o
notebook funcione sem depender de você ter um arquivo à mão. Se preferir usar
seu próprio vídeo, veja a observação logo abaixo dela.

In [ ]:
import cv2
from pathlib import Path

LARGURA, ALTURA = 960, 720
FPS = 20
SEGUNDOS_POR_IMAGEM = 2
N_IMAGENS = 12

random.seed(7)
selecao = random.sample(arquivos, k=min(N_IMAGENS, len(arquivos)))

VIDEO_ENTRADA = "entrada.mp4"
escritor = cv2.VideoWriter(VIDEO_ENTRADA,
                           cv2.VideoWriter_fourcc(*"mp4v"),
                           FPS, (LARGURA, ALTURA))

for caminho in selecao:
    imagem = cv2.resize(cv2.imread(str(caminho)), (LARGURA, ALTURA))
    for _ in range(FPS * SEGUNDOS_POR_IMAGEM):
        escritor.write(imagem)
escritor.release()

duracao = len(selecao) * SEGUNDOS_POR_IMAGEM
print(f"Vídeo criado: {VIDEO_ENTRADA}")
print(f"  {len(selecao)} imagens · {duracao} segundos · {LARGURA}x{ALTURA} @ {FPS} fps")

> **Quer usar seu próprio vídeo?** Substitua a célula acima por:
> ```python
> from google.colab import files
> enviados = files.upload()
> VIDEO_ENTRADA = list(enviados.keys())[0]
> ```

In [ ]:
from collections import Counter, defaultdict

from ultralytics import YOLO

modelo = YOLO(PESOS)

# stream=True processa um quadro por vez, em vez de carregar o vídeo inteiro
# na memória. Sem isso, um vídeo longo derruba o Colab por falta de RAM.
quadros = modelo.track(
    source=VIDEO_ENTRADA,
    conf=0.5,
    save=True,               # grava o vídeo já anotado
    project="runs/video",
    name="teste",
    exist_ok=True,
    stream=True,
    verbose=False,
)

# Para cada ID rastreado, contamos quantas vezes ele foi de cada classe
classe_por_id = defaultdict(Counter)
n_quadros = 0

for quadro in quadros:
    n_quadros += 1
    caixas = quadro.boxes
    if caixas is None or caixas.id is None:
        continue
    ids = caixas.id.int().cpu().tolist()
    cls = caixas.cls.int().cpu().tolist()
    for tid, cid in zip(ids, cls):
        classe_por_id[tid][cid] += 1

print(f"{n_quadros} quadros processados")

In [ ]:
print("=" * 50)
print(f"Animais únicos rastreados: {len(classe_por_id)}")
print("=" * 50)

contagem = Counter()
for tid, votos in classe_por_id.items():
    # A classe mais frequente ao longo do rastro é a resposta final.
    # Assim um quadro isolado mal classificado não estraga o resultado.
    cid = votos.most_common(1)[0][0]
    contagem[modelo.names[cid]] += 1

for nome in modelo.names.values():
    print(f"  {nome:<14} {contagem.get(nome, 0)}")

### Exibindo o vídeo anotado

O Ultralytics grava o vídeo com um codec que nem sempre toca no navegador.
Recodificamos para H.264 com o ffmpeg antes de exibir.

In [ ]:
import base64
from pathlib import Path
from IPython.display import HTML

pasta = Path("runs/video/teste")
gerado = next(p for p in pasta.iterdir() if p.suffix in {".avi", ".mp4"})
print("Vídeo gerado:", gerado)

!ffmpeg -y -loglevel error -i "{gerado}" -vcodec libx264 -crf 23 -pix_fmt yuv420p saida_web.mp4

dados = base64.b64encode(Path("saida_web.mp4").read_bytes()).decode()
HTML(f'<video width=820 controls><source src="data:video/mp4;base64,{dados}" type="video/mp4"></video>')

### Baixando os resultados

Salva os pesos treinados e o vídeo anotado no seu computador.

In [ ]:
from google.colab import files

files.download(PESOS)          # modelo treinado (best.pt)
files.download("saida_web.mp4")  # vídeo anotado

---
# 7. Conclusões

## O que foi feito

Um detector de quatro espécies africanas, treinado por fine-tuning do YOLO26
sobre 1.052 imagens, avaliado em imagens inéditas e aplicado a vídeo com
rastreamento por identidade.

## O que o resultado mostra

O ponto central é que **fine-tuning é muito mais eficiente que treinar do
zero**. Com pouco mais de mil imagens e meia hora de GPU gratuita chega-se a um
detector utilizável, porque o modelo partiu de pesos que já sabiam reconhecer
formas de animais. Todo o aprendizado do Objects365 e do COCO foi reaproveitado.

A matriz de confusão costuma ser o resultado mais revelador: quando há erro, ele
raramente é aleatório. Concentra-se entre espécies visualmente parecidas, o que
é justamente o comportamento esperado de um modelo que aprendeu forma e textura.

## Limitações

- **O dataset é pequeno e específico.** São imagens de reservas sul-africanas,
  em boa parte com o animal bem enquadrado. Imagens de armadilha fotográfica à
  noite, ou animais distantes num rebanho, teriam desempenho bem pior.
- **Só quatro espécies.** Qualquer outro animal ou vira uma das quatro classes,
  ou não é detectado. O modelo não tem como dizer "não sei".
- **O rastreamento usa apenas posição.** Quando dois animais se cruzam, os IDs
  podem trocar. O `botsort.yaml` lida melhor com isso, ao custo de velocidade.

## Possíveis extensões

- Trocar `yolo26s.pt` por `yolo26m.pt` e comparar o ganho de mAP contra o tempo
  de treino
- Exportar para ONNX com `modelo.export(format="onnx")` e rodar sem PyTorch
- Testar em vídeo real de safári e observar a queda de desempenho fora do
  domínio do dataset
- Aplicar o mesmo pipeline a outro dataset oficial da Ultralytics, como
  `construction-ppe.yaml` ou `medical-pills.yaml`